# Step 1 deploy a Gemma

The simplest way:
- go to [Model Garden](https://console.cloud.google.com/vertex-ai/publishers/google/model-garden/gemma) and choose a gemma model to deploy
- **!IMPORTANT!** remember to choose **advanced** deployment settings, and choose a **Public(Shared endpoint)**
- wait a few minutes until the endpoint is provisioned, and then copy the value of the **endpoint ID** as well as **location**
- use the following function to test if your endpoint is well deployed

In [1]:
# Copyright 2020 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# [START aiplatform_predict_custom_trained_model_sample]
from typing import Dict, List, Union

from google.cloud import aiplatform
from google.protobuf import json_format
from google.protobuf.struct_pb2 import Value


def predict_custom_trained_model_sample(
    project: str,
    endpoint_id: str,
    instances: Union[Dict, List[Dict]],
    location: str = "us-central1",
    api_endpoint: str = "us-central1-aiplatform.googleapis.com",
):
    """
    `instances` can be either single instance of type dict or a list
    of instances.
    """
    # The AI Platform services require regional API endpoints.
    client_options = {"api_endpoint": api_endpoint}
    # Initialize client that will be used to create and send requests.
    # This client only needs to be created once, and can be reused for multiple requests.
    client = aiplatform.gapic.PredictionServiceClient(client_options=client_options)
    # The format of each instance should conform to the deployed model's prediction input schema.
    instances = instances if isinstance(instances, list) else [instances]
    instances = [
        json_format.ParseDict(instance_dict, Value()) for instance_dict in instances
    ]
    parameters_dict = {}
    parameters = json_format.ParseDict(parameters_dict, Value())
    endpoint = client.endpoint_path(
        project=project, location=location, endpoint=endpoint_id
    )
    response = client.predict(
        endpoint=endpoint, instances=instances, parameters=parameters
    )
    print("response")
    print(" deployed_model_id:", response.deployed_model_id)
    # The predictions are a google.protobuf.Value representation of the model's predictions.
    predictions = response.predictions
    for prediction in predictions:
        print(" prediction:", prediction)


# [END aiplatform_predict_custom_trained_model_sample]

In [ ]:
PROJECT_NUM = "YOUR PROJECT NUMBER" # this should be a digit-only string
PROJECT_ID = "YOUR PROJECT ID" # this should be a project ID (usually with alphabets and numbers)
ENDPOINT_ID = "YOUR ENDPOINT ID"
LOCATION = "us-central1"

In [ ]:
predict_custom_trained_model_sample(
    project=PROJECT_NUM,
    endpoint_id=ENDPOINT_ID,
    location=LOCATION,
    instances=[
    {
        "prompt": "Hi there",
        "max_tokens": 100,
        "temperature": 1.0,
        "top_p": 1.0,
        "top_k": 1,
    },
]
)

response
 deployed_model_id: 9073727404802834432
 prediction: Prompt:
Hi there
Output:
! I'm a large language model, and I'm here to help you with your requests. Just let me know what you need! 😊
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>


## Step 2 Install liteLLM and set it up with the deployed endpoint

Use pip to install `litellm[proxy]` and `google-adk`and test if it works well with LiteLLM's completion function

In [4]:
!pip install google-adk litellm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.3/239.3 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.1/218.1 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 335.7/335.7 kB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.1/160.1 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.6/201.6 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install 'litellm[proxy]'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 11.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.0/64.0 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.9/187.9 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.9/412.9 kB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.5/95.5 kB 20.3 MB/s eta 0:00:00
   ━━━━━

In [ ]:
from litellm import completion
import os

## set ENV variables
os.environ["VERTEXAI_PROJECT"] = PROJECT_ID
os.environ["VERTEXAI_LOCATION"] = LOCATION

response = completion(
  model=f"vertex_ai/{ENDPOINT_ID}",
  messages=[{ "content": "Hello, how are you?","role": "user"}]
)
print(response)

ModelResponse(id='chatcmpl-121156fd-d0eb-42d7-8f79-dc234751ae09', created=1754644769, model='mg-endpoint-1754489925', object='chat.completion', system_fingerprint=None, choices=[Choices(finish_reason='stop', index=0, message=Message(content="I'm working on a Python script that needs to handle a large dataset", role='assistant', tool_calls=None, function_call=None, provider_specific_fields=None))], usage=Usage(completion_tokens=14, prompt_tokens=6, total_tokens=20, completion_tokens_details=None, prompt_tokens_details=None))


In [2]:
!pip freeze | grep litellm

litellm==1.75.2
litellm-enterprise==0.1.19
litellm-proxy-extras==0.2.16


In [3]:
!pip freeze | grep google-adk

google-adk==1.10.0


In [6]:
!python3 --version

Python 3.11.13


## Step 3 use LiteLlm in ADK

In [ ]:
import os
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
import litellm

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

# --- Example Agent using a fine-tuned Gemini model endpoint ---

# Replace with your fine-tuned model's endpoint resource name
finetuned_gemini_endpoint = f"projects/{PROJECT_NUM}/locations/us-central1/endpoints/{ENDPOINT_ID}"


agent_finetuned_gemini = LlmAgent(
    model=LiteLlm(model=f"vertex_ai/{ENDPOINT_ID}"),
    name="finetuned_gemini_agent",
    instruction="You are a specialized assistant trained on specific data.",
    # ... other agent parameters
)

In [8]:
import asyncio
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

session_service = InMemorySessionService()
session = await session_service.create_session(app_name="my_agent", user_id="test_user")

runner = Runner(
    app_name="my_agent",
    agent=agent_finetuned_gemini,
    session_service=session_service,
)


In [9]:
query = "Hi there, are you doing well today?"
content = types.Content(role="user", parts=[types.Part(text=query)])

for event in runner.run(
    user_id=session.user_id, session_id=session.id, new_message=content
):
    if event.content and event.content.parts:
        if text := "".join(part.text or "" for part in event.content.parts):
            print(f"[{event.author}]: {text}")


[finetuned_gemini_agent]: ```tool_code
print("I am doing well, thank you.")
